In [ ]:
# In the previous lesson we built a RAG pipeline with RAGBase.rag() and saw it fail on the "Olama" typo. The search returned nothing useful, and the LLM had no way to recover.

# The flow that broke:

In [ ]:
# The pipeline is fixed: search, build prompt, LLM.

# def rag(self, query):
#     search_results = self.search(query)
#     prompt = self.build_prompt(query, search_results)
#     answer = self.llm(prompt)
#     return answer

# The LLM is a passenger here, not a driver. It never sees the bad search results, so it can't try again with a corrected query.

In [ ]:
# The difference is about who makes the decisions:

#     With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
#     With an agent, the LLM decides. It chooses which actions to take and when to stop.

# The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [2]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [3]:
MODEL = "openai/gpt-oss-20b"


In [6]:
# ASKING WITHOUT TOOLS
message1 = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model=MODEL,
    messages=message1,
)

response.choices[0].message.content

'Absolutely! Here’s a quick rundown on how you can get on board:\n\n| Step | What to do | Why it matters |\n|------|------------|----------------|\n| **1️⃣ Find the course details** | Locate the course listing on your university portal, LMS, or the online platform where it’s hosted. Note the course code (e.g., *CS\u202f101*), semester/term, and any prerequisite flags. | You’ll need the exact code to search for it and confirm you meet prerequisites. |\n| **2️⃣ Check enrollment status** | Look for a “Register,” “Add,” or “Waitlist” button. If the course shows “Full,” you’ll usually see an option to join a waitlist. | Enrollment can close once a capacity is reached. |\n| **3️⃣ Verify prerequisites** | Ensure you’ve completed or are currently enrolled in the required prerequisite courses (or have been granted an exemption). | Many institutions block registration if prerequisites aren’t satisfied. |\n| **4️⃣ Register (or waitlist)** | Click “Register.” If it’s full, click “Waitlist.” | If y

In [ ]:
# Defining the tool

# First we define a top-level search function that queries the index directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )